In [2]:
try:
    import pyspark
    print("PySpark is already installed!")
except ImportError:
    print("PySpark is not installed. Installing now...")
    !pip install pyspark findspark
    import findspark
    findspark.init()
    import pyspark
    print("PySpark installed and initialized!")

PySpark is already installed!


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

In [4]:
spark = SparkSession.builder.appName("SQLonPySpark").getOrCreate()

In [5]:
# Example DataFrame 1: Employees
data_employees = [
    (1, "Alice", 30, "Engineering"),
    (2, "Bob", 25, "Marketing"),
    (3, "Charlie", 35, "Engineering"),
    (4, "David", 28, "Sales"),
]
columns_employees = ["id", "name", "age", "department"]
df_employees = spark.createDataFrame(data_employees, columns_employees)

In [6]:
# Display the DataFrame and its schema
print("Employee DataFrame:")
df_employees.show()
df_employees.printSchema()

Employee DataFrame:
+---+-------+---+-----------+
| id|   name|age| department|
+---+-------+---+-----------+
|  1|  Alice| 30|Engineering|
|  2|    Bob| 25|  Marketing|
|  3|Charlie| 35|Engineering|
|  4|  David| 28|      Sales|
+---+-------+---+-----------+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- department: string (nullable = true)



In [7]:
# Example DataFrame 2: Salaries
data_salaries = [
    (1, 60000),
    (2, 55000),
    (3, 70000),
    (5, 50000),  # Note: Missing ID 4 for join example
]
columns_salaries = ["id", "salary"]
df_salaries = spark.createDataFrame(data_salaries, columns_salaries)

In [8]:
# Display the DataFrame and its schema
print("\nSalary DataFrame:")
df_salaries.show()
df_salaries.printSchema()


Salary DataFrame:
+---+------+
| id|salary|
+---+------+
|  1| 60000|
|  2| 55000|
|  3| 70000|
|  5| 50000|
+---+------+

root
 |-- id: long (nullable = true)
 |-- salary: long (nullable = true)



In [9]:
# Selecting specific columns using string names
df_employees.select("name", "department").show()

# Selecting columns using the col() function for more robust referencing
from pyspark.sql.functions import col
df_employees.select(col("name"), col("department")).show()

# You can also access columns like attributes (less recommended for complex names)
df_employees.select(df_employees.name, df_employees.department).show()

+-------+-----------+
|   name| department|
+-------+-----------+
|  Alice|Engineering|
|    Bob|  Marketing|
|Charlie|Engineering|
|  David|      Sales|
+-------+-----------+

+-------+-----------+
|   name| department|
+-------+-----------+
|  Alice|Engineering|
|    Bob|  Marketing|
|Charlie|Engineering|
|  David|      Sales|
+-------+-----------+

+-------+-----------+
|   name| department|
+-------+-----------+
|  Alice|Engineering|
|    Bob|  Marketing|
|Charlie|Engineering|
|  David|      Sales|
+-------+-----------+



In [10]:
# Filtering based on a single condition using filter() or where()
df_employees.filter(df_employees["department"] == "Engineering").show()
df_employees.where(col("department") == "Engineering").show() # where() is an alias for filter()

# Filtering with multiple conditions using logical AND (&)
df_employees.filter((col("age") >= 25) & (col("department") == "Marketing")).show()

# Filtering with multiple conditions using logical OR (|)
df_employees.filter((col("department") == "Marketing") | (col("department") == "Sales")).show()

+---+-------+---+-----------+
| id|   name|age| department|
+---+-------+---+-----------+
|  1|  Alice| 30|Engineering|
|  3|Charlie| 35|Engineering|
+---+-------+---+-----------+

+---+-------+---+-----------+
| id|   name|age| department|
+---+-------+---+-----------+
|  1|  Alice| 30|Engineering|
|  3|Charlie| 35|Engineering|
+---+-------+---+-----------+

+---+----+---+----------+
| id|name|age|department|
+---+----+---+----------+
|  2| Bob| 25| Marketing|
+---+----+---+----------+

+---+-----+---+----------+
| id| name|age|department|
+---+-----+---+----------+
|  2|  Bob| 25| Marketing|
|  4|David| 28|     Sales|
+---+-----+---+----------+



In [11]:
# Sorting by a single column in ascending order (default)
df_employees.orderBy("age").show()
df_employees.sort(col("age")).show() # sort() is an alias for orderBy()

# Sorting by a single column in descending order
df_employees.orderBy(col("age").desc()).show()

# Sorting by multiple columns
df_employees.orderBy(["department", col("age").desc()]).show

+---+-------+---+-----------+
| id|   name|age| department|
+---+-------+---+-----------+
|  2|    Bob| 25|  Marketing|
|  4|  David| 28|      Sales|
|  1|  Alice| 30|Engineering|
|  3|Charlie| 35|Engineering|
+---+-------+---+-----------+

+---+-------+---+-----------+
| id|   name|age| department|
+---+-------+---+-----------+
|  2|    Bob| 25|  Marketing|
|  4|  David| 28|      Sales|
|  1|  Alice| 30|Engineering|
|  3|Charlie| 35|Engineering|
+---+-------+---+-----------+

+---+-------+---+-----------+
| id|   name|age| department|
+---+-------+---+-----------+
|  3|Charlie| 35|Engineering|
|  1|  Alice| 30|Engineering|
|  4|  David| 28|      Sales|
|  2|    Bob| 25|  Marketing|
+---+-------+---+-----------+



<bound method DataFrame.show of DataFrame[id: bigint, name: string, age: bigint, department: string]>

In [12]:
df_employees.limit(2).show()

+---+-----+---+-----------+
| id| name|age| department|
+---+-----+---+-----------+
|  1|Alice| 30|Engineering|
|  2|  Bob| 25|  Marketing|
+---+-----+---+-----------+



In [13]:
df_employees.select("department").distinct().show()

+-----------+
| department|
+-----------+
|Engineering|
|  Marketing|
|      Sales|
+-----------+



In [14]:
from pyspark.sql import functions as F

# Grouping by 'department' and counting the number of employees in each department
df_employees.groupBy("department").count().withColumnRenamed("count", "employee_count").show()

# Grouping by 'department' and calculating the average age
df_employees.groupBy("department").agg(F.avg("age").alias("average_age")).show()

# Grouping by 'department' and calculating the minimum and maximum age
df_employees.groupBy("department").agg(F.min("age").alias("min_age"), F.max("age").alias("max_age")).show()

+-----------+--------------+
| department|employee_count|
+-----------+--------------+
|Engineering|             2|
|  Marketing|             1|
|      Sales|             1|
+-----------+--------------+

+-----------+-----------+
| department|average_age|
+-----------+-----------+
|Engineering|       32.5|
|  Marketing|       25.0|
|      Sales|       28.0|
+-----------+-----------+

+-----------+-------+-------+
| department|min_age|max_age|
+-----------+-------+-------+
|Engineering|     30|     35|
|  Marketing|     25|     25|
|      Sales|     28|     28|
+-----------+-------+-------+



In [15]:
# INNER JOIN: Returns rows when there is a match in both DataFrames
df_employees.join(df_salaries, "id", "inner").show()

# LEFT JOIN: Returns all rows from the left DataFrame and matching rows from the right.
# If there's no match in the right, it fills with nulls.
df_employees.join(df_salaries, "id", "left").show()

# RIGHT JOIN: Returns all rows from the right DataFrame and matching rows from the left.
# If there's no match in the left, it fills with nulls.
df_employees.join(df_salaries, "id", "right").show()

# FULL OUTER JOIN: Returns all rows from both DataFrames.
# If there's no match, it fills with nulls for the missing side.
df_employees.join(df_salaries, "id", "full").show()

+---+-------+---+-----------+------+
| id|   name|age| department|salary|
+---+-------+---+-----------+------+
|  1|  Alice| 30|Engineering| 60000|
|  2|    Bob| 25|  Marketing| 55000|
|  3|Charlie| 35|Engineering| 70000|
+---+-------+---+-----------+------+

+---+-------+---+-----------+------+
| id|   name|age| department|salary|
+---+-------+---+-----------+------+
|  1|  Alice| 30|Engineering| 60000|
|  2|    Bob| 25|  Marketing| 55000|
|  3|Charlie| 35|Engineering| 70000|
|  4|  David| 28|      Sales|  NULL|
+---+-------+---+-----------+------+

+---+-------+----+-----------+------+
| id|   name| age| department|salary|
+---+-------+----+-----------+------+
|  1|  Alice|  30|Engineering| 60000|
|  2|    Bob|  25|  Marketing| 55000|
|  5|   NULL|NULL|       NULL| 50000|
|  3|Charlie|  35|Engineering| 70000|
+---+-------+----+-----------+------+

+---+-------+----+-----------+------+
| id|   name| age| department|salary|
+---+-------+----+-----------+------+
|  1|  Alice|  30|Engine

In [17]:
# Create a duplicate of the employees DataFrame for demonstration
df_employees_duplicate = spark.createDataFrame(data_employees, columns_employees)

# UNION ALL: Combines rows from both DataFrames, including duplicates
df_union_all = df_employees.select("name").unionAll(df_employees_duplicate.select("name"))
df_union_all.show()
print(f"Number of rows in UNION ALL: {df_union_all.count()}")

# UNION DISTINCT: Combines rows and removes duplicates using distinct()
df_union_distinct = df_employees.select("name").unionAll(df_employees_duplicate.select("name")).distinct()
df_union_distinct.show()
print(f"Number of rows in UNION DISTINCT: {df_union_distinct.count()}")

+-------+
|   name|
+-------+
|  Alice|
|    Bob|
|Charlie|
|  David|
|  Alice|
|    Bob|
|Charlie|
|  David|
+-------+

Number of rows in UNION ALL: 8
+-------+
|   name|
+-------+
|    Bob|
|  Alice|
|Charlie|
|  David|
+-------+

Number of rows in UNION DISTINCT: 4


In [18]:
df_employees_with_country = df_employees.withColumn("country", lit("India"))
df_employees_with_country.show()

from pyspark.sql.functions import concat_ws
df_employees_with_greeting = df_employees.withColumn("greeting", concat_ws(", ", lit("Hello"), col("name")))
df_employees_with_greeting.show()

+---+-------+---+-----------+-------+
| id|   name|age| department|country|
+---+-------+---+-----------+-------+
|  1|  Alice| 30|Engineering|  India|
|  2|    Bob| 25|  Marketing|  India|
|  3|Charlie| 35|Engineering|  India|
|  4|  David| 28|      Sales|  India|
+---+-------+---+-----------+-------+

+---+-------+---+-----------+--------------+
| id|   name|age| department|      greeting|
+---+-------+---+-----------+--------------+
|  1|  Alice| 30|Engineering|  Hello, Alice|
|  2|    Bob| 25|  Marketing|    Hello, Bob|
|  3|Charlie| 35|Engineering|Hello, Charlie|
|  4|  David| 28|      Sales|  Hello, David|
+---+-------+---+-----------+--------------+



In [19]:
from pyspark.sql.functions import when
df_updated_age = df_employees.withColumn("age", when(col("department") == "Engineering", col("age") + 1).otherwise(col("age")))
df_updated_age.show()

+---+-------+---+-----------+
| id|   name|age| department|
+---+-------+---+-----------+
|  1|  Alice| 31|Engineering|
|  2|    Bob| 25|  Marketing|
|  3|Charlie| 36|Engineering|
|  4|  David| 28|      Sales|
+---+-------+---+-----------+



In [20]:
df_employees_dropped = df_employees.drop("age")
df_employees_dropped.show()

df_employees_dropped_multiple = df_employees.drop("age", "department")
df_employees_dropped_multiple.show()

+---+-------+-----------+
| id|   name| department|
+---+-------+-----------+
|  1|  Alice|Engineering|
|  2|    Bob|  Marketing|
|  3|Charlie|Engineering|
|  4|  David|      Sales|
+---+-------+-----------+

+---+-------+
| id|   name|
+---+-------+
|  1|  Alice|
|  2|    Bob|
|  3|Charlie|
|  4|  David|
+---+-------+

